Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
os.chdir("../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Loading and Preprocessing Data

In [3]:
# Download Data
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time'])
#full_df.head(5)

In [4]:
# Calculates normalized hourly residuals
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df)

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [5]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

Random Forest Regression

In [6]:
def compute_mae(y, yhat):
    """given predicted and observed values, computes mean absolute error"""
    return np.mean(np.abs(yhat - y))

def forecast(model, exog, y_init):
    """given a trained model, exogenous features, and initial AR term, makes forecasting predictions"""
    yhat = []
    Xi_te = np.hstack([y_init, exog[0]])[None,:]
    for i in range(len(exog)-1):
        yhat_i = model.predict(Xi_te)[0]
        yhat.append(yhat_i)
        Xi_te = np.hstack([yhat_i, exog[i+1]])[None,:]
    yhat.append(model.predict(Xi_te)[0])
    return np.array(yhat)
def weekly_forecast(model, exog, y_init):
    """given a trained model exogenous features, and initial AR term, makes a series of 1-week-out forecasts"""
    yhat = []
    for i, yi in enumerate(y_init):
        exog_i = exog[168*i:168*(i+1),:]
        if exog_i.shape[0] < 1:
            break
        yhat.append(forecast(model, exog_i, yi))
    return np.hstack(yhat)
    

In [7]:
import pickle
from sklearn.ensemble import RandomForestRegressor
from mapie.regression import MapieRegressor

file_path = 'ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/rfr_no_mapie_pkl.pkl'  
with open(file_path, 'rb') as f:
    x = pickle.load(f)
    # identify model parameters and current model weights from pickle file
    print("Model parameters:", x.get_params())
    print("Feature importances:", x.feature_importances_)
    # save model parameters and weights as dictionaries
    parameters = x.get_params()
    weights = x.feature_importances_
    # train new RFR model, and get validation performance
    def get_rfr_mae(sd):
        mod = RandomForestRegressor(n_estimators=parameters["n_estimators"], max_depth=parameters["max_depth"], random_state=sd)
        mod.fit(X_tr, y_tr)
        pred_val = weekly_forecast(mod, exog_val, y_init_val)
        return compute_mae(y_val, pred_val) 
        
    # define range of random seeds being searched
    seed = list(range(1,760))
    grid_search = pd.DataFrame(columns=['seed','mae'])
    
    # perform search to identify the best performing model seed -- Q will saving the model w this seed set change the performance at all
    for sd in seed:
        mae = get_rfr_mae(sd)
        #print(mae)
        params = {'seed': sd, 'mae':mae} 
        params_df = pd.DataFrame([params])
        grid_search = pd.concat([grid_search, params_df], ignore_index=True)
        
    # display best hyperparameters based on grid search
    grid_rfr = grid_search.sort_values('mae').head(1)
    print(grid_rfr)
    # best hyperparameters
    sd = grid_rfr['seed'].values[0]
    # train model and get predictions
    mod_rfr = RandomForestRegressor(random_state=sd)
    mod_rfr.fit(X_tr, y_tr)
    
    mapie = MapieRegressor(mod_rfr, method="naive")
    mapie.fit(X_tr, y_tr)
    pred_mapie, pred_ci = ForecastingHelpers.weekly_forecast(yp_val.index, mapie, exog_val, y_init_val, 0.05)
    print('MAE:', ForecastingHelpers.compute_mae(y_val, pred_mapie))
    
    # save seed manipulated model to pickle file
    import pickle
    filename = "ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/rfr_seed_manip_pkl.pkl"
    with open(filename, 'wb') as file:
        pickle.dump(mod_rfr, file)


Model parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 8, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False}
Feature importances: [9.39982657e-01 1.15581826e-03 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 4.97536047e-06 2.82380390e-03
 5.60327459e-02]


/tmp/ipykernel_170416/1923560168.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grid_search = pd.concat([grid_search, params_df], ignore_index=True)


    seed       mae
751  752  0.054385
MAE: 0.040914825959327675


In [8]:
import onnx
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/rfr_no_mapie.onnx")

In [9]:
import pickle
file_name = "ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/rfr_seed_manip_pkl.pkl"
with open(file_name, 'rb') as f:
    sess = pickle.load(f)
    

In [10]:
def forecast(session, exog, y_init):
    """given a trained model, exogenous features, and initial AR term, makes forecasting predictions"""
    yhat = []
    y_ci = []
    Xi_te = np.hstack([y_init, exog[0]])[None,:]
    for i in range(len(exog)-1):
        yhat_i = sess.predict(Xi_te.astype(np.float32))
        yhat.append(yhat_i)
        Xi_te = np.hstack([yhat_i, exog[i+1]])[None,:]
    yhat_i = sess.predict(Xi_te.astype(np.float32))
    yhat.append(yhat_i)
    return np.array(yhat)

def weekly_forecast(indexes, session, exog, y_init):
    """given a trained model exogenous features, and initial AR term, makes a series of 1-week-out forecasts"""
    yhat = []
    for i, yi in enumerate(y_init):
        exog_i = exog[168*i:168*(i+1),:]
        if exog_i.shape[0] < 1:
            break
        y_hat_i = forecast(session, exog_i, yi)
        yhat.append(y_hat_i)
    mapie_hat = pd.DataFrame(np.vstack(yhat).reshape(-1))
    return mapie_hat.values.ravel()
    

In [11]:
# Inspect attributes
print(dir(sess))
print("Model parameter names:", sess._get_metadata_request())
print("Model parameters:", sess.get_params())
print("Feature importances:", sess.feature_importances_)
print("# Feature:", sess.feature_importances_.size)

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__slotnames__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_build_request_for_signature', '_check_feature_names', '_check_n_features', '_compute_oob_predictions', '_compute_partial_dependence_recursion', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_estimator_type', '_get_default_requests', '_get_doc_link', '_get_estimators_indices', '_get_metadata_request', '_get_oob_predictions', '_get_param_names', '_get_tags', '_make_estimator', '_more_tags', '_n_samples', '_n_sample

In [12]:
preds_pkl = weekly_forecast(yp_te.index, sess, exog_te, y_init_te)
print(preds_pkl)

[-0.13215294 -0.12819726 -0.15668337 ...  0.03394648 -0.0024666
 -0.00247807]


In [13]:
# plotting testing
print('MAE:', ForecastingHelpers.compute_mae(y_test, preds_pkl))

MAE: 0.05058537969603207


Explore model info??

In [14]:
import onnx

def get_tree_ensemble_attributes(model_path):
    """Loads an ONNX model and extracts attributes from a TreeEnsembleRegressor node."""
    model = onnx.load(model_path)
    nodesInfo = {}
    for node in model.graph.node:
        if node.op_type == "TreeEnsembleRegressor":
            print(f"Found TreeEnsembleRegressor node: {node.name}")

            # The attributes contain the tree structure
            for attr in node.attribute:
                print(f"\nAttribute name: {attr.name}")
                # For simplicity, we only print a few key attributes.
                # The full tree structure is encoded here.
                if attr.name == "n_targets":
                    print(f"  Number of targets: {attr.i}")
                elif attr.name == "nodes_nodeids":
                    print(f"  Node feature ids: {attr.ints[:]}")
                elif attr.name == "nodes_hitrates":
                    print(f"  Node hit rates: {attr.floats[:]}")
                elif attr.name == "nodes_values":
                    print(f"  Nodes values: {attr.floats[:]}")
                elif attr.name == "target_weights":
                    print(f"  Target weights: {attr.floats[:]}")
                # You would need to parse these attributes to reconstruct the trees
    else:
        print("No TreeEnsembleRegressor node found.")


In [15]:
#onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/rfr_no_mapie.onnx")
# Replace 'your_model.onnx' with the path to your ONNX file
get_tree_ensemble_attributes("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/rfr_no_mapie.onnx")


Found TreeEnsembleRegressor node: TreeEnsembleRegressor

Attribute name: n_targets
  Number of targets: 1

Attribute name: nodes_falsenodeids

Attribute name: nodes_featureids

Attribute name: nodes_hitrates
  Node hit rates: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0

In [16]:
for node in onnx_model.graph.node:
    print("name=%r type=%r input=%r output=%r" % (
        node.name, node.op_type, node.input, node.output))

name='TreeEnsembleRegressor' type='TreeEnsembleRegressor' input=['X'] output=['variable']


In [17]:
from onnx2torch import convert

ModuleNotFoundError: No module named 'onnx2torch'